---
# Commande pour le rendu : quarto render main.ipynb --execute
# Instructions pour quarto
title: "Rendu final"
format:
  html:
    code-fold: true
    embed-resources: true
---

<!-- Header stylé -->
<div style="background: linear-gradient(90deg, #4e54c8, #8f94fb); padding: 30px; border-radius: 12px; color: white; text-align: center; margin-bottom: 20px;">
  <h1 style="margin: 0; font-size: 3em;">🚧 Page en Construction 🚧</h1>
  <p style="font-size: 1.\; margin-top: 10px;">
    Le rendu final sera ici !
  </p>
  <a href="https://github.com/gaybrice/projet-python-ds" target="_blank" 
     style="color: #ffd700; font-weight: bold; text-decoration: none; font-size: 1.1em;">
     🔗 Lien vers le GitHub
  </a>
</div>

# Traitement des données IDFM

Cette page présente le code permettant d'obtenir un dataset contenant la liste des arrêts de transport en commun d'Île de France avec :
- le nombre de passage de bus, trains et métros à chaque arrêt
- le nombre de lignes desservant chaque arrêt
- la localisation de chaque arrêt

## Récupération des données
Pour cela nous commençons par récupérer les données issues de l'API Île-de-France Mobilités.

In [ ]:
import script.download_data as download_data
import pandas as pd

file_names = download_data.get_IDFM_data_path()

# open only usefull df :
usefull_keys = ["routes", "trips", "stop_times", "stops", "calendar"]

idfm = {k: pd.read_csv(f) for k, f in file_names.items() if k in usefull_keys}

print("Les fichiers suivants on été ouverts :")
for k in idfm.keys():
    print(f"- {k}")

Dossier cache trouvé, pas de téléchargement.
None
Available GTFS files:
- calendar: cache/idfm/calendar.csv
- stop_times: cache/idfm/stop_times.csv
- pathways: cache/idfm/pathways.csv
- ticketing_deep_links: cache/idfm/ticketing_deep_links.csv
- object_codes_extension: cache/idfm/object_codes_extension.csv
- transfers: cache/idfm/transfers.csv
- trips: cache/idfm/trips.csv
- routes: cache/idfm/routes.csv
- booking_rules: cache/idfm/booking_rules.csv
- calendar_dates: cache/idfm/calendar_dates.csv
- stops: cache/idfm/stops.csv
- stop_extensions: cache/idfm/stop_extensions.csv
- agency: cache/idfm/agency.csv
dict_keys(['calendar', 'stop_times', 'pathways', 'ticketing_deep_links', 'object_codes_extension', 'transfers', 'trips', 'routes', 'booking_rules', 'calendar_dates', 'stops', 'stop_extensions', 'agency'])


/tmp/ipykernel_237397/4261714513.py:14: DtypeWarning:

Columns (12,13) have mixed types. Specify dtype option on import or set low_memory=False.

/tmp/ipykernel_237397/4261714513.py:14: DtypeWarning:

Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.



In [ ]:
# Affichage des dataframes
for name, df in idfm.items():
    print(f"=== {name} ===")
    print(df.shape)
    print(df.head(3).to_string(index=False))
    print()

=== calendar ===
(1479, 10)
service_id  monday  tuesday  wednesday  thursday  friday  saturday  sunday  start_date  end_date
    IDFM:1       1        1          1         0       1         1       1    20251212  20251231
 IDFM:1029       0        0          0         0       1         0       0    20251212  20251219
 IDFM:1040       0        0          0         0       0         1       0    20251213  20251220

=== stop_times ===
(12132256, 14)
                           trip_id arrival_time departure_time  start_pickup_drop_off_window  end_pickup_drop_off_window    stop_id  stop_sequence  pickup_type  drop_off_type  local_zone_id  stop_headsign  timepoint pickup_booking_rule_id drop_off_booking_rule_id
IDFM:stif:local-175995-C00562-5495     12:34:00       12:34:00                           NaN                         NaN IDFM:10228              0            0              1            NaN            NaN          1                    NaN                      NaN
IDFM:stif:local-17599

In [ ]:
# on remplace stop_id par parent_station pour les arrêts parents dans stop_times et stops
# le but est d'obtenir une ligne par arret commercial (initialement: la gare de bus et la gare RER d'un même arret partagent le même nom d'arret commercial mais ont des stop_id différents)

print(f"unique stop_id before replacement: {idfm['stops']['stop_id'].nunique()} in stops, {idfm['stop_times']['stop_id'].nunique()} in stop_times")

# les arrets parents n'ont pas de parent_station, on remplace les NA par leur propre stop_id
nb_na_before = idfm["stops"]["parent_station"].isna().sum()
idfm["stops"].fillna({"parent_station": idfm["stops"]["stop_id"]}, inplace=True)
print(f"NA parent_station : before {nb_na_before}, after {idfm['stops']['parent_station'].isna().sum()}")

# on remplace stop_id par parent_station dans stop_times
idfm["stop_times"] = idfm["stop_times"].merge(
    idfm["stops"][["stop_id", "parent_station"]], on="stop_id", how="left"
).drop(columns=["stop_id"]).rename(columns={"parent_station": "stop_id"})

# on ne conserve que les arrêts parents dans stops
idfm["stops"] = idfm["stops"][idfm["stops"]["parent_station"] == idfm["stops"]["stop_id"]].reset_index(drop=True)


print(f"unique stop_id after replacement: {idfm['stops']['stop_id'].nunique()} in stops, {idfm['stop_times']['stop_id'].nunique()} in stop_times")

unique stop_id before replacement: 53688 in stops, 35639 in stop_times
NA parent_station : before 15374, after 0
unique stop_id after replacement: 15374 in stops, 15374 in stop_times


In [ ]:
# On ne conserve que les services ayant un jour donné pour calculer un nombre de trajet sur une journée en semaine

# on prend le prochain lundi à partir d'aujourd'hui au format YYYYMMDD
from datetime import datetime, timedelta
today = datetime.today()
next_monday = today + timedelta(days=(7 - today.weekday()) % 7)
next_monday_str = next_monday.strftime("%Y%m%d")
print(f"Travail avec le jour: {next_monday_str}")

next_monday_str = "20251215"  # reproductibilité # TODO :


print(len(idfm["calendar"][(idfm["calendar"]["monday"] == 1)]))
print(len(idfm["calendar"][(idfm["calendar"]["monday"] == 1) & (idfm["calendar"]["start_date"] <= int(next_monday_str)) & (idfm["calendar"]["end_date"] >= int(next_monday_str)) ]))
services_one_day =  idfm["calendar"][(idfm["calendar"]["monday"] == 1) & (idfm["calendar"]["start_date"] <= int(next_monday_str)) & (idfm["calendar"]["end_date"] >= int(next_monday_str)) ]["service_id"]
trips_one_day = idfm["trips"].merge(services_one_day, on="service_id", how="inner")
print(f"Redution du nombre de trajets en ne conservant qu'un jour : {len(idfm['trips'])} -> {len(trips_one_day)}")

Travail avec le jour: 20251215
621
468
Redution du nombre de trajets en ne conservant qu'un jour : 535147 -> 122860


In [ ]:
# Pour chaque stop_id, et chaque route_id, on calcul le nombre d'arrets par jour à ce stop
# voir le pdf docu_gtfs.pdf, section 3.4.5 page 17 pour la structure des tables
stop_times_one_day = idfm["stop_times"].merge(trips_one_day[["trip_id", "route_id"]], on="trip_id", how="inner")
stop_route_counts = stop_times_one_day.groupby(["stop_id", "route_id"]).size().reset_index(name="nb_stops_per_day")
print(f"Nombre de paires (stop_id, route_id) uniques le sur le jour {next_monday_str} : {len(stop_route_counts)}")
print(stop_route_counts.head(10).to_string(index=False))

Nombre de paires (stop_id, route_id) uniques le sur le jour 20251215 : 35015
    stop_id    route_id  nb_stops_per_day
IDFM:411281 IDFM:C01843               284
IDFM:411284 IDFM:C01843               284
IDFM:411318 IDFM:C01746                34
IDFM:411327 IDFM:C01737                32
IDFM:411330 IDFM:C01737                32
IDFM:411333 IDFM:C01737                32
IDFM:411339 IDFM:C01739                19
IDFM:411343 IDFM:C01739                19
IDFM:411346 IDFM:C01739                29
IDFM:411349 IDFM:C01739                19


In [ ]:
# on ajoute les infos des arrêts
stop_route_counts = (stop_route_counts.merge(idfm["stops"][["stop_id", "stop_name", "stop_lat", "stop_lon", "zone_id"]], on="stop_id", how="left")
                      .merge(idfm["routes"][["route_id", "route_short_name", "route_type", "route_color"]], on="route_id", how="left")
)
stop_route_counts.head(10)

,stop_id,route_id,nb_stops_per_day,stop_name,stop_lat,stop_lon,zone_id,route_short_name,route_type,route_color
0,IDFM:411281,IDFM:C01843,284,Lycée Henri Sellier,48.916327,2.514983,NaN,T4,0,DFAF47
1,IDFM:411284,IDFM:C01843,284,Rougemont Chanteloup,48.930712,2.515149,NaN,T4,0,DFAF47
2,IDFM:411318,IDFM:C01746,34,Beauvais,49.426247,2.088346,NaN,TER,2,AAAAAA
3,IDFM:411327,IDFM:C01737,32,Précy-sur-Oise,49.203511,2.376149,NaN,H,2,84653D
4,IDFM:411330,IDFM:C01737,32,Saint-Leu-d'Esserent,49.213869,2.417613,NaN,H,2,84653D
5,IDFM:411333,IDFM:C01737,32,Boran-sur-Oise,49.169392,2.360668,NaN,H,2,84653D
6,IDFM:411339,IDFM:C01739,19,Trie-Château,49.283233,1.820375,NaN,J,2,CEC73D
7,IDFM:411343,IDFM:C01739,19,Liancourt-Saint-Pierre,49.220605,1.905423,NaN,J,2,CEC73D
8,IDFM:411346,IDFM:C01739,29,Chaumont-en-Vexin,49.261285,1.872898,NaN,J,2,CEC73D
9,IDFM:411349,IDFM:C01739,19,Lavilletertre,49.202382,1.920579,NaN,J,2,CEC73D


# Sous partie controle des probleme des bus

In [ ]:
# controle : on affiche les lignes avec le plus de stops par jour
# Résultats cohérents sauf pour certaines lignes de bus pour lesquelles on un un arret avec un nombre de passage beaucoup plus élevé que les autres arrets de la même ligne
# Exemples : 91, 92, TVM
# FUN est le funiculaire de Montmartre, avec peu d'arrets mais beaucoup de passages par jour

df = stop_route_counts.groupby(["route_id", "route_short_name"])["nb_stops_per_day"].agg(["median", "max"]).reset_index()
df["dif_max_median"] = df["max"] - df["median"]
df.sort_values(by="max", ascending=False).head(10)
#df[df["dif_max_median"] > 10].sort_values(by="dif_max_median", ascending=False)


,route_id,route_short_name,median,max,dif_max_median
1543,IDFM:C02666,C1,2148.0,2153,5.0
639,IDFM:C01122,91,363.0,1381,1018.0
850,IDFM:C01385,FUN,1110.0,1110,0.0
640,IDFM:C01123,92,367.0,1071,704.0
591,IDFM:C01071,TVM,509.0,1018,509.0
617,IDFM:C01098,62,317.0,948,631.0
742,IDFM:C01248,258,272.0,929,657.0
848,IDFM:C01383,13,872.0,872,0.0
646,IDFM:C01132,103,429.0,858,429.0
841,IDFM:C01376,6,797.0,797,0.0


In [ ]:
print(stop_route_counts.sort_values("nb_stops_per_day", ascending=False).head(50).to_string(index=False))

    stop_id    route_id  nb_stops_per_day                                       stop_name  stop_lat  stop_lon  zone_id route_short_name  route_type route_color
 IDFM:69636 IDFM:C02666              2153                                     La Végétale 48.741797  2.478499      NaN               C1           6      3C91DC
 IDFM:74103 IDFM:C02666              2149                                Limeil-Brévannes 48.752103  2.472085      NaN               C1           6      3C91DC
 IDFM:69591 IDFM:C02666              2148                                      Villa Nova 48.735107  2.464819      NaN               C1           6      3C91DC
 IDFM:69685 IDFM:C02666              2148                                        Valenton 48.748710  2.471732      NaN               C1           6      3C91DC
 IDFM:69884 IDFM:C02666              2142                                   Pointe du Lac 48.769031  2.464355      NaN               C1           6      3C91DC
 IDFM:71139 IDFM:C01122              138

Dans ce tableau on voit que tous les arrets de la ligne 1 ont 978 passages par jour. Ce qui est cohérent. Pour les lignes TVM, 91 et 92 en revanche, un seul arret cumul plus de 1000 passages de bus, alors que les autres en ont beaucoup moins. Il y a donc un problème avec ces arrêts.

Comme on le voit ci-dessous, le TVM a un stop pour lequel il y 1 fois plus de passage que les autres. C'est la même chose pour la ligne 91 

In [ ]:
print(stop_route_counts[stop_route_counts["route_short_name"] == "TVM"].sort_values("nb_stops_per_day", ascending=False).to_string(index=False))


    stop_id    route_id  nb_stops_per_day                                         stop_name  stop_lat  stop_lon  zone_id route_short_name  route_type route_color
IDFM:412677 IDFM:C01071              1018   La Haye aux Moines / Préfecture du Val-de-Marne 48.784896  2.448309      NaN              TVM           3      216EB4
IDFM:493232 IDFM:C01071               509                                Marcelin Berthelot 48.769038  2.421502      NaN              TVM           3      216EB4
 IDFM:69779 IDFM:C01071               509                                      Victor Basch 48.758351  2.392863      NaN              TVM           3      216EB4
 IDFM:69772 IDFM:C01071               509                        Carrefour de la Résistance 48.757384  2.388582      NaN              TVM           3      216EB4
IDFM:494371 IDFM:C01071               509                                         Pompadour 48.773019  2.434827      NaN              TVM           3      216EB4
IDFM:493231 IDFM:C01071     

In [ ]:
# Pour debug en regadrant les stops d'une ligne particuliere
merged = idfm["stop_times"].merge(trips_one_day, on="trip_id", how="inner")
# print(merged[(merged["stop_id"] == "IDFM:71139") & (merged["route_id"] == "IDFM:C01122")].count())

print(merged[(merged["stop_id"] == "IDFM:71139") & (merged["route_id"] == "IDFM:C01122") & (merged["direction_id"] == 0)].sort_values("arrival_time"))


                                                   trip_id arrival_time  \
1467864  IDFM:RATP:193594-C01122-COU_RATP_091LVT15Semai...     05:49:00   
1467865  IDFM:RATP:193594-C01122-COU_RATP_091LVT15Semai...     05:51:00   
1467866  IDFM:RATP:193594-C01122-COU_RATP_091LVT15Semai...     05:52:00   
1467868  IDFM:RATP:193594-C01122-COU_RATP_091LVT15Semai...     05:57:00   
1467946  IDFM:RATP:193594-C01122-COU_RATP_091LVT15Semai...     05:58:00   
...                                                    ...          ...   
1469068  IDFM:RATP:193594-C01122-COU_RATP_091LVT15Semai...     24:53:00   
1470273  IDFM:RATP:193594-C01122-COU_RATP_091LVT15Semai...     24:56:00   
1470274  IDFM:RATP:193594-C01122-COU_RATP_091LVT15Semai...     24:58:00   
1470275  IDFM:RATP:193594-C01122-COU_RATP_091LVT15Semai...     24:59:00   
1470277  IDFM:RATP:193594-C01122-COU_RATP_091LVT15Semai...     25:03:00   

        departure_time  start_pickup_drop_off_window  \
1467864       05:49:00                     

# Suite traitement


In [ ]:
stop_route_counts["route_type"].value_counts()

route_type
3    33750
2      575
1      405
0      278
6        5
7        2
Name: count, dtype: int64

In [ ]:
# calcul du nombre de passage par type de transport
passage_par_arret = (stop_route_counts.groupby(["stop_id", "stop_name", "stop_lat", "stop_lon", "route_type"]) 
                    ["nb_stops_per_day"].sum().reset_index())
passage_par_arret = (passage_par_arret[passage_par_arret["route_type"].isin([0, 1, 2, 3])].reset_index(drop=True)
                    .pivot_table(index=["stop_id", "stop_name", "stop_lat", "stop_lon"], columns="route_type", values="nb_stops_per_day", fill_value=0))
# rename columns
passage_par_arret = passage_par_arret.rename(columns={
    0: "nb_tramway_per_day",
    1: "nb_metro_per_day",
    2: "nb_train_per_day",
    3: "nb_bus_per_day"
})
passage_par_arret = passage_par_arret.reset_index()
passage_par_arret

route_type,stop_id,stop_name,stop_lat,stop_lon,nb_tramway_per_day,nb_metro_per_day,nb_train_per_day,nb_bus_per_day
0,IDFM:411281,Lycée Henri Sellier,48.916327,2.514983,284.0,0.0,0.0,0.0
1,IDFM:411284,Rougemont Chanteloup,48.930712,2.515149,284.0,0.0,0.0,0.0
2,IDFM:411318,Beauvais,49.426247,2.088346,0.0,0.0,34.0,0.0
3,IDFM:411327,Précy-sur-Oise,49.203511,2.376149,0.0,0.0,32.0,0.0
4,IDFM:411330,Saint-Leu-d'Esserent,49.213869,2.417613,0.0,0.0,32.0,0.0
...,...,...,...,...,...,...,...,...
13750,IDFM:74370,Rue du Lavoir,48.629995,2.058604,0.0,0.0,0.0,3.0
13751,IDFM:74372,Hauts Flouviers,48.749094,2.370086,0.0,0.0,0.0,279.0
13752,IDFM:74373,Collège Lucie Aubrac,48.960136,2.352398,0.0,0.0,0.0,177.0
13753,IDFM:74375,Mediathèque,49.035847,2.460687,0.0,0.0,0.0,14.0


In [ ]:
print(passage_par_arret["stop_name"].nunique())
print(passage_par_arret["stop_id"].nunique())

10405
13755


In [ ]:
passage_par_arret.keys()

Index(['stop_id', 'stop_name', 'stop_lat', 'stop_lon', 'nb_tramway_per_day',
       'nb_metro_per_day', 'nb_train_per_day', 'nb_bus_per_day'],
      dtype='object', name='route_type')

# Affichage sur une carte

In [ ]:
# exemple d'utilisation de load_fonds_carte qui récupère les bordures des départements et des communes d'IDF
import geopandas as gpd
import matplotlib.pyplot as plt
from script.download_fond_carte import load_fonds_carte

coms, deps = load_fonds_carte(crs=4326, force_download=False)

# Listes filtrées de la "petite couronne"
PETITE_COURONNE = [75, 92, 93, 94]
coms_pc = coms[coms["INSEE_DEP"].astype(int).isin(PETITE_COURONNE)].copy()
deps_pc = deps[deps["INSEE_DEP"].astype(int).isin(PETITE_COURONNE)].copy()

Téléchargement des bordures de communes / départements mis en cache


In [ ]:
# Plotly version : affichage des arrêts de tramway de la petite couronne avec >100 passages/jour
import plotly.graph_objects as go

# filtrer les gares tramway (threshold 100)
gares_a_afficher = passage_par_arret
gares_gdf = gpd.GeoDataFrame(
    gares_a_afficher,
    geometry=gpd.points_from_xy(gares_a_afficher["stop_lon"], gares_a_afficher["stop_lat"]),
    crs=4326,
)
gares_pc_only = gpd.sjoin(gares_gdf, coms_pc[['INSEE_COM', 'geometry']], how='inner', predicate='within')

# préparer traces pour les contours (departements + communes)
def build_line_coords(geoms):
    lons, lats = [], []
    for g in geoms:
        if g is None or g.is_empty:
            continue
        if g.geom_type == "MultiLineString" or g.geom_type == "MultiPolygon":
            for part in g:
                x, y = part.exterior.xy if hasattr(part, "exterior") else ([], [])
                if x:
                    lons += list(x) + [None]
                    lats += list(y) + [None]
        else:
            if g.geom_type == "Polygon":
                x, y = g.exterior.xy
                lons += list(x) + [None]
                lats += list(y) + [None]
            else:  # LineString
                x, y = g.xy
                lons += list(x) + [None]
                lats += list(y) + [None]
    return lons, lats

deps_lons, deps_lats = build_line_coords(deps_pc.boundary.geometry)
coms_lons, coms_lats = build_line_coords(coms_pc.boundary.geometry)

# points
pts = gares_pc_only
sizes = (pts.get("nb_train_per_day", 0).fillna(0) + 10).clip(lower=6)  # taille des markers
hover = pts["stop_name"] + "<br>tram/day: " + pts["nb_tramway_per_day"].astype(int).astype(str) \
        + "<br>train/day: " + pts.get("nb_train_per_day", 0).fillna(0).astype(int).astype(str)

# centre et zoom basique
minx, miny, maxx, maxy = coms_pc.total_bounds
center = {"lon": float((minx + maxx) / 2), "lat": float((miny + maxy) / 2)}

fig = go.Figure()

# departement boundaries (thicker, black)
fig.add_trace(go.Scattermapbox(
    lon=deps_lons, lat=deps_lats, mode="lines",
    line=dict(width=1.2, color="black"), hoverinfo="none", name="Départements"
))

# commune boundaries (thin, grey)
fig.add_trace(go.Scattermapbox(
    lon=coms_lons, lat=coms_lats, mode="lines",
    line=dict(width=0.5, color="gray"), hoverinfo="none", name="Communes"
))
# ensure we pass plain lists/arrays of the same length to plotly (drop any invalid coords)
pts_clean = pts.dropna(subset=["stop_lon", "stop_lat"]).copy()
if pts_clean.empty:
    print("Aucun point valide à afficher après suppression des coordonnées manquantes.")
else:
    lons_pts = pts_clean["stop_lon"].astype(float).tolist()
    lats_pts = pts_clean["stop_lat"].astype(float).tolist()

    # sizes / colors as plain lists
    sizes_arr = (pts_clean.get("nb_train_per_day", 0).fillna(0) + 10).clip(lower=6).astype(float).tolist()
    color_arr = pts_clean.get("nb_train_per_day", 0).fillna(0).astype(float).tolist()

    # prepare hover text
    hover_arr = (pts_clean["stop_name"].astype(str)
                    + "<br>tram/day: " + pts_clean["nb_tramway_per_day"].fillna(0).astype(int).astype(str)
                    + "<br>train/day: " + pts_clean.get("nb_train_per_day", 0).fillna(0).astype(int).astype(str)).tolist()

    fig.add_trace(go.Scattermapbox(
        lon=lons_pts, lat=lats_pts,
        mode="markers",
        marker=dict(
                size=sizes_arr,
                color=color_arr,
                colorscale="inferno",
                showscale=True,
                colorbar=dict(title="train/day")
            ),
        text=hover_arr,
        hoverinfo="text",
        name="Gares tramway"
    ))

    fig.update_layout(
        mapbox=dict(
            style="open-street-map",
            center=center,
            zoom=11
        ),
        margin=dict(l=0, r=0, t=30, b=0),
        legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01),
        title=f"Gares tramway petite couronne (>100 passages/jour) — {len(pts_clean)} gares affichées"
    )

    fig.show()

In [ ]:
# Plotly version : affichage des arrêts de tramway de la petite couronne avec >100 passages/jour
import plotly.graph_objects as go

# filtrer les gares tramway (threshold 100)
gares_a_afficher = passage_par_arret[passage_par_arret["nb_tramway_per_day"] > 100].copy()
if gares_a_afficher.empty:
    print("Aucune gare de tramway avec >100 passages/jour dans les données.")
else:
    gares_gdf = gpd.GeoDataFrame(
        gares_a_afficher,
        geometry=gpd.points_from_xy(gares_a_afficher["stop_lon"], gares_a_afficher["stop_lat"]),
        crs=4326,
    )
    gares_pc_only = gpd.sjoin(gares_gdf, coms_pc[['INSEE_COM', 'geometry']], how='inner', predicate='within')

    # préparer traces pour les contours (departements + communes)
    def build_line_coords(geoms):
        lons, lats = [], []
        for g in geoms:
            if g is None or g.is_empty:
                continue
            if g.geom_type == "MultiLineString" or g.geom_type == "MultiPolygon":
                for part in g:
                    x, y = part.exterior.xy if hasattr(part, "exterior") else ([], [])
                    if x:
                        lons += list(x) + [None]
                        lats += list(y) + [None]
            else:
                if g.geom_type == "Polygon":
                    x, y = g.exterior.xy
                    lons += list(x) + [None]
                    lats += list(y) + [None]
                else:  # LineString
                    x, y = g.xy
                    lons += list(x) + [None]
                    lats += list(y) + [None]
        return lons, lats

    deps_lons, deps_lats = build_line_coords(deps_pc.boundary.geometry)
    coms_lons, coms_lats = build_line_coords(coms_pc.boundary.geometry)

    # points
    pts = gares_pc_only
    sizes = (pts.get("nb_train_per_day", 0).fillna(0) + 10).clip(lower=6)  # taille des markers
    hover = pts["stop_name"] + "<br>tram/day: " + pts["nb_tramway_per_day"].astype(int).astype(str) \
            + "<br>train/day: " + pts.get("nb_train_per_day", 0).fillna(0).astype(int).astype(str)

    # centre et zoom basique
    minx, miny, maxx, maxy = coms_pc.total_bounds
    center = {"lon": float((minx + maxx) / 2), "lat": float((miny + maxy) / 2)}

    fig = go.Figure()

    # departement boundaries (thicker, black)
    fig.add_trace(go.Scattermapbox(
        lon=deps_lons, lat=deps_lats, mode="lines",
        line=dict(width=1.2, color="black"), hoverinfo="none", name="Départements"
    ))

    # commune boundaries (thin, grey)
    fig.add_trace(go.Scattermapbox(
        lon=coms_lons, lat=coms_lats, mode="lines",
        line=dict(width=0.5, color="gray"), hoverinfo="none", name="Communes"
    ))
    # ensure we pass plain lists/arrays of the same length to plotly (drop any invalid coords)
    pts_clean = pts.dropna(subset=["stop_lon", "stop_lat"]).copy()
    if pts_clean.empty:
        print("Aucun point valide à afficher après suppression des coordonnées manquantes.")
    else:
        lons_pts = pts_clean["stop_lon"].astype(float).tolist()
        lats_pts = pts_clean["stop_lat"].astype(float).tolist()

        # sizes / colors as plain lists
        sizes_arr = (pts_clean.get("nb_train_per_day", 0).fillna(0) + 10).clip(lower=6).astype(float).tolist()
        color_arr = pts_clean.get("nb_train_per_day", 0).fillna(0).astype(float).tolist()

        # prepare hover text
        hover_arr = (pts_clean["stop_name"].astype(str)
                     + "<br>tram/day: " + pts_clean["nb_tramway_per_day"].fillna(0).astype(int).astype(str)
                     + "<br>train/day: " + pts_clean.get("nb_train_per_day", 0).fillna(0).astype(int).astype(str)).tolist()

        fig.add_trace(go.Scattermapbox(
            lon=lons_pts, lat=lats_pts,
            mode="markers",
            marker=dict(
                    size=sizes_arr,
                    color=color_arr,
                    colorscale="inferno",
                    showscale=True,
                    colorbar=dict(title="train/day")
                ),
            text=hover_arr,
            hoverinfo="text",
            name="Gares tramway"
        ))

        fig.update_layout(
            mapbox=dict(
                style="open-street-map",
                center=center,
                zoom=11
            ),
            margin=dict(l=0, r=0, t=30, b=0),
            legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01),
            title=f"Gares tramway petite couronne (>100 passages/jour) — {len(pts_clean)} gares affichées"
        )

        fig.show()

/tmp/ipykernel_237397/3622487597.py:55: DeprecationWarning:

*scattermapbox* is deprecated! Use *scattermap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/

/tmp/ipykernel_237397/3622487597.py:61: DeprecationWarning:

*scattermapbox* is deprecated! Use *scattermap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/

/tmp/ipykernel_237397/3622487597.py:82: DeprecationWarning:

*scattermapbox* is deprecated! Use *scattermap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/

